# DSC@UIT 2026 — Task 1: LegalIR
**Team: SGU_LegalMind** · Codabench: https://www.codabench.org/competitions/17715/

Pipeline truy vấn văn bản pháp luật tiếng Việt:

```
question ─┬─► BM25 (chunk)   ─┐
          └─► dense (chunk)  ─┴─► RRF fuse ─► top-N doc ─► cross-encoder rerank ─► top-5
```

## Layout Drive thực tế

```
[Shared with me] UIT DS Challenge/        ← owner: phatleh.224@gmail.com
└── Task 1/
    ├── LegalR - Public Test/             ← DỮ LIỆU (notebook tự dò tới đây)
    │     train.json · public-official.json · selected-contexts.zip
    ├── Scoring-Program-Task-LegalIR/     ← scoring.py chính thức của BTC
    ├── SourceCode/                       ← notebook này
    ├── Vòng 1 - Warm up/
    └── Vòng 2 - PublicTest/

[My Drive] DSC2026_Task1/artifacts/       ← notebook GHI vào đây (~1.5GB)
```

Artifact ghi vào **My Drive của bạn**, không ghi vào thư mục chia sẻ của nhóm — vừa
tránh làm rối thư mục chung, vừa tránh bị người khác xoá mất 1.5GB kết quả encode.

## ⚠ Bắt buộc làm 1 lần trước khi chạy

`drive.mount()` **chỉ mount My Drive**. Thư mục `UIT DS Challenge` đang ở
**"Shared with me"** nên Colab *không* thấy nó → phải tạo shortcut:

> drive.google.com → **Shared with me** → chuột phải **UIT DS Challenge**
> → **Organise** → **Add shortcut to Drive** → chọn **My Drive**

Sau bước này notebook **tự dò** ra đường dẫn, không cần sửa gì. Nếu vẫn
không ra, cell "Tự dò thư mục dữ liệu" sẽ in hướng dẫn và bạn gán tay biến `RAW`.

## Cách dùng
1. Làm bước shortcut ở trên.
2. Chạy tuần tự các Stage. Mỗi stage **cache artifact ra Drive** và tự bỏ qua nếu đã có
   → session Colab chết giữa đường thì chạy lại không mất việc đã làm.
3. **Lần đầu để `SMOKE = True`** (Stage 0 config) để chạy hết pipeline trên corpus nhỏ
   trong ~5 phút, xác nhận không lỗi. Sau đó đặt `SMOKE = False` và chạy thật.

## Mốc so sánh
Baseline TF-IDF toàn văn bản (truncate 20k ký tự), đo trên 500 câu train:

| | recall@5 | recall@20 | recall@100 |
|---|---|---|---|
| TF-IDF doc-level | **0.486** | 0.698 | 0.870 |

Khoảng cách `@100 = 0.87` vs `@5 = 0.486` cho thấy phần lớn điểm mất nằm ở **ranking**,
không phải retrieval → reranker là bước ăn điểm mạnh nhất.

## Ràng buộc cuộc thi (đọc từ `scoring.py` chính thức, không phải từ mô tả)
- **Tối đa 5 document_id / câu hỏi.** Vượt quá ở bất kỳ câu nào → câu đó nhận **0** cả
  Recall lẫn Precision.
- **Recall không cắt top-5** — code BTC chỉ *chặn* ở `len <= 5`, rồi lấy giao của
  **toàn bộ** danh sách nộp. Nên nộp **đúng 5 id** là tối ưu cho Recall.
- **Lệch số câu hoặc thiếu qid → `raise Exception` / `TypeError` → sập TOÀN BỘ bài nộp**,
  không phải chỉ mất điểm câu đó. Validator ở Stage 9 chặn cả hai.
- **id phải là string.** Code chấm dùng `set(truth) & set(pred)`; nộp `123` (int) sẽ
  không khớp `"123"` (str) → 0 điểm dù dự đoán đúng.
- **Recall là metric chính**, Precision chỉ dùng tie-break → **luôn nộp đủ 5 id**.
- **Tổng tham số toàn hệ thống < 4 tỷ** (gồm cả embedding layer).
- **Cấm mọi API.** Chỉ model open-source tự host.
- **Phải đăng ký model với BTC trước khi dùng** — xem cuối notebook.

## Model dùng trong notebook này

| Vai | Model | Tham số | Context |
|---|---|---|---|
| bi-encoder | `bkai-foundation-models/vietnamese-bi-encoder` | 135M | 256 |
| reranker | `BAAI/bge-reranker-v2-m3` | 568M | 512 |
| | **Tổng** | **0.70B** < 4B | |


---
## Stage 0 — Config & môi trường

In [ ]:
#@title Config
import os, sys

ON_COLAB = os.path.isdir('/content')
PROFILE  = 'v1'

# ---- Nơi GHI artifact: My Drive CỦA BẠN ------------------------------------
# Không ghi vào thư mục được chia sẻ của nhóm: artifact ~1.5GB và người khác
# có thể xoá mất. Thư mục này notebook tự tạo.
ART_ROOT = '/content/drive/MyDrive/DSC2026_Task1' if ON_COLAB else './work'
ART      = f'{ART_ROOT}/artifacts'
SCRATCH  = '/content/scratch' if ON_COLAB else './scratch'   # đĩa tạm, KHÔNG phải Drive

# ---- Nơi ĐỌC dữ liệu: cell sau tự dò ---------------------------------------
# Nếu thư mục dữ liệu đang ở "Shared with me" thì Colab KHÔNG mount phần đó.
# Cell "Tự dò thư mục dữ liệu" xử lý mọi trường hợp.
DATA_FILES = ('train.json', 'public-official.json')
RAW = None   # cell "dò dữ liệu" sẽ điền

# ---- Chế độ ----------------------------------------------------------------
# SMOKE=True: corpus 1500 doc + 200 query -> chạy hết pipeline ~5 phút để test.
# Đặt False khi chạy thật. Validator sẽ TỪ CHỐI tạo submission nếu SMOKE=True.
SMOKE = True

# ---- Chunking --------------------------------------------------------------
CHUNK_CHARS        = 1000   # khớp MAX_SEQ của bi-encoder (~4 ký tự/token)
CHUNK_OVERLAP      = 200
MAX_CHUNKS_PER_DOC = 300    # chặn văn bản khổng lồ (max corpus: 6M ký tự!)

# ---- Retrieval -------------------------------------------------------------
BM25_K1, BM25_B = 0.9, 0.4  # tuned cho document dài
CAND_CHUNKS     = 1000      # số chunk lấy ra trước khi gộp về document
CAND_DOCS       = 100       # số doc mỗi nhánh đưa vào RRF
RRF_K           = 60
RERANK_TOP      = 50       # số doc đưa vào cross-encoder
TOPK_SUBMIT     = 5         # RÀNG BUỘC CỨNG của BTC

# ---- Models (phải đăng ký với BTC trước khi dùng) --------------------------
BIENCODER = 'bkai-foundation-models/vietnamese-bi-encoder'   # 135M
RERANKER  = 'BAAI/bge-reranker-v2-m3'    # 568M   -> tổng 0.70B < 4B
MAX_SEQ   = 256       # token/chunk cho bi-encoder
FT_BIENCODER_PATH = f'{ART}/models/biencoder-ft-{PROFILE}'   # Stage 8 ghi ra đây
USE_FINETUNED = False       # bật sau khi chạy Stage 8

# ---- Eval ------------------------------------------------------------------
N_VAL = 1000
SEED  = 42

ENC_SHARD    = 50000   # encode theo shard, checkpoint từng shard ra Drive
ENC_BATCH    = 128
RERANK_MAXLEN = 512
RERANK_BATCH  = 32

print('ON_COLAB =', ON_COLAB, '| SMOKE =', SMOKE)
print('ART      =', ART)

In [ ]:
#@title Mount Drive + cài dependency
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    !pip -q install sentence-transformers==3.0.1 pyarrow 2>&1 | tail -1

import os
for d in (ART, f'{ART}/emb', f'{ART}/models', SCRATCH):
    os.makedirs(d, exist_ok=True)

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
if torch.cuda.is_available():
    print('VRAM %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))

In [ ]:
#@title Tự dò thư mục dữ liệu
# Thư mục "UIT DS Challenge" được người khác chia sẻ nên nằm trong "Shared with me",
# mà drive.mount() CHỈ mount My Drive -> đường dẫn hardcode sẽ không tồn tại.
# Cell này dò qua cả 3 nơi Colab có thể phơi dữ liệu ra.
import os, glob

def find_data_dir(roots, max_depth=6):
    """Trả về thư mục đầu tiên chứa đủ DATA_FILES."""
    for root in roots:
        if not os.path.isdir(root):
            continue
        base = root.rstrip('/').count('/')
        for dirpath, dirnames, filenames in os.walk(root):
            if dirpath.count('/') - base >= max_depth:
                dirnames[:] = []
                continue
            # bỏ thư mục ẩn / rác macOS cho nhanh
            dirnames[:] = [d for d in dirnames
                           if not d.startswith('.') and d != '__MACOSX']
            if all(f in filenames for f in DATA_FILES):
                return dirpath
    return None

if RAW is None:
    ROOTS = ['/content/drive/MyDrive',
             '/content/drive/.shortcut-targets-by-id',   # nơi shortcut trỏ tới
             '/content/drive/Shareddrives',
             '/content'] if ON_COLAB else ['.', '..']
    print('đang dò...', flush=True)
    RAW = find_data_dir(ROOTS)

assert RAW, (
    'KHÔNG tìm thấy dữ liệu.\n\n'
    'Nguyên nhân thường gặp: thư mục "UIT DS Challenge" đang ở "Shared with me",\n'
    'mà Colab không mount phần đó. Cách sửa (làm 1 lần):\n'
    '  Mở drive.google.com -> Shared with me -> chuột phải "UIT DS Challenge"\n'
    '  -> Organise -> Add shortcut to Drive -> chọn My Drive.\n'
    'Rồi chạy lại cell này.\n\n'
    'Hoặc gán tay: RAW = "/content/drive/MyDrive/.../LegalR - Public Test"')

print('RAW =', RAW)
for f in DATA_FILES:
    p = f'{RAW}/{f}'
    print(f'  {f:26s} {os.path.getsize(p)/1e6:8.2f} MB')

ZIP = f'{RAW}/selected-contexts.zip'
if os.path.exists(ZIP):
    print(f'  {"selected-contexts.zip":26s} {os.path.getsize(ZIP)/1e6:8.2f} MB')
else:
    print('  selected-contexts.zip      KHÔNG THẤY — Stage 1 sẽ tìm folder đã giải nén')

---
## Stage 1 — Build corpus

Đọc 8.532 file `context_*.json` → **một** file parquet.

Hai vấn đề dữ liệu được xử lý ở đây:
- Zip giải nén **lồng hai lớp** (`selected-contexts/selected-contexts/...`) → dùng glob đệ quy.
- **1.125/8.532 văn bản (13%) thiếu field `name`** → lấy lại tiêu đề từ slug trong `link`.
  Tiêu đề là tín hiệu rất mạnh (chứa loại văn bản + chủ đề + năm), bỏ 13% là mất điểm thật.

Giải nén vào `/content` (đĩa local) chứ **không** vào Drive — ghi 8.532 file nhỏ lên Drive
mất hàng giờ.

In [ ]:
#@title Build corpus.parquet
import json, re, glob, os, zipfile, time
import pandas as pd

CORPUS_PQ = f'{ART}/corpus.parquet'

def _title_from_link(link):
    slug = (link or '').rstrip('/').rsplit('/', 1)[-1]
    slug = re.sub(r'\.aspx$', '', slug, flags=re.I)
    return slug.replace('-', ' ').strip()

def build_corpus():
    raw_dir  = globals().get('CTX_DIR', f'{SCRATCH}/contexts')
    zip_path = globals().get('CTX_ZIP', f'{RAW}/selected-contexts.zip')
    files = glob.glob(f'{raw_dir}/**/context_*.json', recursive=True)
    if not files:
        # thử folder đã giải nén sẵn cạnh dữ liệu gốc
        files = glob.glob(f'{RAW}/**/context_*.json', recursive=True)
    if not files:
        assert os.path.exists(zip_path), f'Không thấy {zip_path} — hãy upload lên Drive.'
        print('Giải nén zip vào đĩa tạm...', flush=True)
        os.makedirs(raw_dir, exist_ok=True)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(raw_dir)
        files = glob.glob(f'{raw_dir}/**/context_*.json', recursive=True)
    print(f'{len(files)} file context', flush=True)

    rows, n_noname = [], 0
    for i, fp in enumerate(files):
        with open(fp, encoding='utf-8') as f:
            d = json.load(f)
        title = (d.get('name') or '').replace('-', ' ').strip()
        if not title:
            n_noname += 1
            title = _title_from_link(d.get('link', ''))
        # gộp mọi whitespace: passage có rất nhiều '\r\n\n  ' rác.
        # Làm sớm để regex tách 'Điều N' ở Stage 2 hoạt động ổn định.
        passage = re.sub(r'\s+', ' ', d.get('passage') or '').strip()
        rows.append({'doc_id': str(d['id']), 'title': title,
                     'link': d.get('link', ''), 'passage': passage})
        if (i + 1) % 2000 == 0:
            print(f'  {i+1}/{len(files)}', flush=True)

    df = pd.DataFrame(rows).drop_duplicates('doc_id').reset_index(drop=True)
    print(f'-> {len(df)} doc | thiếu name (đã vá từ link): {n_noname}')
    df.to_parquet(CORPUS_PQ, index=False)
    return df

t0 = time.time()
if os.path.exists(CORPUS_PQ):
    corpus = pd.read_parquet(CORPUS_PQ)
    print(f'load cache: {len(corpus)} doc')
else:
    corpus = build_corpus()

print('%.1fs' % (time.time() - t0))
print('passage chars: mean %d | p50 %d | max %d' % (
    corpus.passage.str.len().mean(), corpus.passage.str.len().median(),
    corpus.passage.str.len().max()))
corpus.head(3)[['doc_id', 'title']]

In [ ]:
#@title Load train / public + val split cố định
import json, random

with open(f'{RAW}/train.json', encoding='utf-8') as f:
    train_raw = json.load(f)
with open(f'{RAW}/public-official.json', encoding='utf-8') as f:
    public_raw = json.load(f)

# answer có thể là int hoặc str tuỳ file -> luôn ép str
QUESTIONS = {qid: v['question'] for qid, v in train_raw.items()}
GOLD = {qid: set(map(str, v['answer'])) for qid, v in train_raw.items()}
PUBLIC = {qid: v['question'] for qid, v in public_raw.items()}

# Split cố định theo seed + sorted key -> mọi session cho cùng một val set.
# Đây là điều kiện bắt buộc để so sánh các thí nghiệm với nhau.
all_qids = sorted(train_raw.keys())
random.Random(SEED).shuffle(all_qids)
VAL_QIDS   = all_qids[:N_VAL]
TRAIN_QIDS = all_qids[N_VAL:]

print(f'train {len(TRAIN_QIDS)} | val {len(VAL_QIDS)} | public {len(PUBLIC)}')

# kiểm tra gold có nằm trong corpus không
corpus_ids = set(corpus.doc_id)
missing = {g for q in all_qids for g in GOLD[q]} - corpus_ids
print('gold doc thiếu trong corpus:', len(missing))

In [ ]:
#@title SMOKE mode — thu nhỏ corpus + query để test pipeline
# Giữ toàn bộ gold doc của các query được chọn, rồi thêm doc ngẫu nhiên làm distractor.
# Nhờ vậy recall trong SMOKE vẫn có ý nghĩa (chỉ là bài toán dễ hơn nhiều).
if SMOKE:
    rng = random.Random(SEED)
    VAL_QIDS   = VAL_QIDS[:200]
    TRAIN_QIDS = TRAIN_QIDS[:800]
    keep = {g for q in VAL_QIDS + TRAIN_QIDS for g in GOLD[q]}
    pool = [d for d in corpus.doc_id if d not in keep]
    keep |= set(rng.sample(pool, min(1500, len(pool))))
    corpus = corpus[corpus.doc_id.isin(keep)].reset_index(drop=True)
    PUBLIC = dict(list(PUBLIC.items())[:200])
    print(f'SMOKE: corpus {len(corpus)} doc | val {len(VAL_QIDS)} | train {len(TRAIN_QIDS)}')
else:
    print('chế độ THẬT — corpus %d doc' % len(corpus))

---
## Stage 2 — Chunking theo cấu trúc pháp quy

Passage trung bình **41.000 ký tự** mà bi-encoder chỉ nhận ~256 token (~1.000 ký tự).
Không chunk là không dùng được dense retrieval.

Ba quyết định thiết kế ở đây:
- **Cắt theo `Điều N`** thay vì cắt cứng theo độ dài — ranh giới ngữ nghĩa của văn bản
  pháp luật nằm đúng ở đó. Điều ngắn thì gộp lại, Điều dài thì cắt tiếp có overlap.
- **Prepend tiêu đề vào mọi chunk.** Câu hỏi rất ngắn (trung bình 20 từ) nên tiêu đề
  văn bản là tín hiệu định chủ đề cực mạnh.
- **Chunk đầu tiên = tiêu đề + phần mở đầu**, nơi chứa số hiệu, cơ quan ban hành, trích yếu.
- **Cap `MAX_CHUNKS_PER_DOC`**: nếu không, vài văn bản 6M ký tự sẽ chiếm phần lớn index
  và bóp méo điểm `max` khi gộp chunk→doc.

In [ ]:
#@title Build chunks.parquet
import re, os, time
import pandas as pd

CHUNKS_PQ = f'{ART}/chunks_c{CHUNK_CHARS}{"_smoke" if SMOKE else ""}.parquet'

DIEU_RE = re.compile(r'(?=Điều\s+\d+\s*[\.\:])')

def _split_long(s, size, overlap):
    """Cắt s thành các đoạn <= size ký tự, ưu tiên cắt ở ranh giới câu."""
    out, i = [], 0
    while i < len(s):
        j = min(i + size, len(s))
        if j < len(s):
            # lùi về dấu chấm gần nhất trong 200 ký tự cuối cho khỏi cắt giữa câu
            cut = s.rfind('. ', i + size - 200, j)
            if cut > i:
                j = cut + 1
        out.append(s[i:j].strip())
        if j >= len(s):
            break
        i = max(j - overlap, i + 1)
    return [c for c in out if c]

def chunk_doc(title, passage):
    """-> list các đoạn text (chưa gắn tiêu đề)."""
    pieces = [p.strip() for p in DIEU_RE.split(passage) if p.strip()]
    if not pieces:
        pieces = [passage]

    merged = []
    buf = ''
    for p in pieces:
        if len(p) > CHUNK_CHARS:
            if buf:
                merged.append(buf); buf = ''
            merged.extend(_split_long(p, CHUNK_CHARS, CHUNK_OVERLAP))
        elif len(buf) + len(p) + 1 <= CHUNK_CHARS:
            buf = f'{buf} {p}'.strip()
        else:
            merged.append(buf); buf = p
    if buf:
        merged.append(buf)

    # chunk mở đầu: chứa số hiệu / cơ quan ban hành / trích yếu
    head = passage[:CHUNK_CHARS].strip()
    if head and (not merged or merged[0][:200] != head[:200]):
        merged.insert(0, head)
    return merged[:MAX_CHUNKS_PER_DOC]

def build_chunks(corpus):
    rows = []
    for n, r in enumerate(corpus.itertuples(index=False)):
        for c in chunk_doc(r.title, r.passage):
            # tiêu đề đi kèm mọi chunk -> chunk lẻ vẫn biết mình thuộc văn bản nào
            rows.append((r.doc_id, f'{r.title}. {c}'))
        if (n + 1) % 1000 == 0:
            print(f'  {n+1}/{len(corpus)} doc -> {len(rows)} chunk', flush=True)
    df = pd.DataFrame(rows, columns=['doc_id', 'text'])
    df.to_parquet(CHUNKS_PQ, index=False)
    return df

t0 = time.time()
if os.path.exists(CHUNKS_PQ):
    chunks = pd.read_parquet(CHUNKS_PQ)
    print('load cache')
else:
    chunks = build_chunks(corpus)
print('%d chunk | %.1fs' % (len(chunks), time.time() - t0))

# map chunk -> doc dạng số nguyên, dùng suốt phần sau
DOC_IDS = corpus.doc_id.tolist()
DOC2IX  = {d: i for i, d in enumerate(DOC_IDS)}
import numpy as np
CHUNK2DOC = np.array([DOC2IX[d] for d in chunks.doc_id], dtype=np.int32)
print('chunk/doc trung bình: %.1f' % (len(chunks) / len(corpus)))
chunks.text.str.len().describe()[['mean', '50%', 'max']]

---
## Stage 3 — Hạ tầng đánh giá

**Làm cái này trước khi tối ưu bất cứ gì.** `public-official.json` không có đáp án và chỉ
được nộp 10 lần/ngày, nên mọi quyết định kỹ thuật phải ra từ val set này — miễn phí, không
giới hạn số lần thử.

Công thức theo đúng BTC: `Recall = mean_q |gold ∩ pred| / |gold|` với `pred` cắt ở 5 doc.

Luôn in cả `@5`, `@20`, `@100` vì chúng chẩn đoán hai bệnh khác nhau:
- `@100` thấp → **retriever** yếu (văn bản đúng không vào nổi danh sách ứng viên)
- `@100` cao mà `@5` thấp → **ranking** yếu (tìm ra rồi nhưng không đẩy lên top)

In [ ]:
#@title Metrics + experiment log
import numpy as np, pandas as pd, os, datetime

def official_score(sub, truth):
    """Bản sao CHÍNH XÁC của scoring.py mà BTC dùng trên Codabench.

    sub: {qid: {'answer': [id]}} — đúng định dạng file nộp
    truth: {qid: [id]}

    Ba chi tiết trong code của BTC mà đọc mô tả không thấy:
    1. Recall KHÔNG cắt top-5. Nó chỉ *chặn* ở len<=5: quá 5 id -> câu đó 0 điểm.
       Nộp đúng 5 id là tối ưu cho Recall.
    2. Số lượng key phải khớp tuyệt đối, lệch là raise Exception -> TOÀN BỘ bài nộp
       lỗi, không phải chỉ mất điểm câu đó.
    3. Thiếu key -> len(None) -> TypeError -> cũng sập toàn bài.
    """
    y_pred = {k: v['answer'] for k, v in sub.items()}
    if len(y_pred) != len(truth):
        raise Exception('Samples in predict not match with reference')
    ok = lambda k: 0 < len(y_pred[k]) <= TOPK_SUBMIT
    recall = np.mean([len(set(truth[k]) & set(y_pred[k])) / len(truth[k])
                      if ok(k) else 0 for k in truth])
    precision = np.mean([len(set(truth[k]) & set(y_pred[k])) / len(y_pred[k])
                         if ok(k) else 0 for k in y_pred])
    return {'recall': float(recall), 'precision': float(precision)}

def evaluate(preds, gold, ks=(5, 20, 100)):
    """preds: {qid: [doc_id đã xếp hạng]} -> recall@k chẩn đoán + điểm chính thức.

    recall@20/@100 KHÔNG phải metric của cuộc thi — chúng để chẩn đoán:
      @100 thấp        -> retriever yếu (văn bản đúng không vào nổi danh sách)
      @100 cao, @5 thấp -> ranking yếu (tìm ra rồi mà không đẩy lên top)
    """
    out = {}
    for k in ks:
        out[f'recall@{k}'] = float(np.mean([
            len(gold[q] & set(preds.get(q, [])[:k])) / len(gold[q]) for q in gold]))
    # điểm chính thức: dựng đúng shape file nộp rồi chấm bằng code của BTC
    sub = {q: {'answer': list(preds.get(q, [])[:TOPK_SUBMIT])} for q in gold}
    off = official_score(sub, {q: list(gold[q]) for q in gold})
    out['OFFICIAL_recall'] = off['recall']
    out['OFFICIAL_precision'] = off['precision']
    return out

def show(name, preds, gold=None, qids=None):
    gold = gold or {q: GOLD[q] for q in (qids or VAL_QIDS)}
    m = evaluate(preds, gold)
    print(name.ljust(28), ' '.join(f'{k}={v:.4f}' for k, v in m.items()))
    return m

LOG_CSV = f'{ART}/experiments.csv'

def log_exp(name, metrics, note=''):
    """Ghi lại mọi thí nghiệm. Bảng này là thứ bạn cần để chọn cấu hình cho private
    test và để viết paper nếu vào top 10 — đừng bỏ qua."""
    row = {'time': datetime.datetime.now().isoformat(timespec='seconds'),
           'profile': PROFILE, 'run': name, 'smoke': SMOKE, 'note': note, **metrics}
    df = pd.DataFrame([row])
    if os.path.exists(LOG_CSV):
        df = pd.concat([pd.read_csv(LOG_CSV), df], ignore_index=True)
    df.to_csv(LOG_CSV, index=False)
    return df.tail(5)

print('ok')

In [ ]:
#@title Gộp chunk -> doc bằng MAX, rồi lấy top-N doc
def chunks_to_docs(chunk_scores, chunk_top_ix, topn=CAND_DOCS):
    """Điểm doc = MAX điểm các chunk của nó (không phải mean).

    Gold là "văn bản CHỨA thông tin cần thiết" -> một chunk khớp là đủ. Lấy mean sẽ
    phạt oan các văn bản dài mà chỉ một Điều liên quan.
    """
    best = {}
    for ix, s in zip(chunk_top_ix, chunk_scores):
        d = CHUNK2DOC[ix]
        if s > best.get(d, -1e30):
            best[d] = s
    ranked = sorted(best.items(), key=lambda x: -x[1])[:topn]
    return [DOC_IDS[d] for d, _ in ranked], [s for _, s in ranked]

print('ok')

---
## Stage 4 — BM25 (chunk-level)

Không cần GPU — **chạy stage này trong session CPU** để tiết kiệm quota GPU, đó là tài
nguyên khan hiếm nhất của bạn.

Cài đặt BM25 dạng sparse matrix (`W @ q`) thay vì `rank_bm25`: với ~500k chunk thì
`rank_bm25` chậm tới mức không dùng được, còn sparse matmul chấm 1 query mất vài chục ms.

BM25 bù đúng chỗ dense yếu: thuật ngữ pháp lý hiếm (*"Kiểm ngư viên trung cấp"*,
*"hành lang an toàn kỹ thuật anten"*) mà embedding thường trượt hẳn.

In [ ]:
#@title BM25 sparse
import numpy as np, scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer

class SparseBM25:
    """BM25 Okapi tiền tính thành ma trận trọng số -> truy vấn = 1 phép matmul sparse."""

    def __init__(self, k1=BM25_K1, b=BM25_B, min_df=3, max_features=400_000):
        self.k1, self.b = k1, b
        self.vec = CountVectorizer(lowercase=True, min_df=min_df,
                                   max_features=max_features, dtype=np.int32)

    def fit(self, texts):
        X = self.vec.fit_transform(texts).tocsc()
        n, _ = X.shape
        dl = np.asarray(X.sum(axis=1)).ravel()
        avgdl = dl.mean()
        df = np.diff(X.indptr)
        idf = np.log(1 + (n - df + 0.5) / (df + 0.5)).astype(np.float32)

        X = X.tocoo()
        tf = X.data.astype(np.float32)
        denom = tf + self.k1 * (1 - self.b + self.b * dl[X.row] / avgdl)
        data = idf[X.col] * tf * (self.k1 + 1) / denom
        self.W = sp.csr_matrix((data, (X.row, X.col)), shape=(n, len(idf),))
        self.W = self.W.T.tocsr()      # (vocab, n_chunk) -> query nhanh hơn
        return self

    def search(self, queries, topk=CAND_CHUNKS):
        Q = self.vec.transform(queries)
        Q.data[:] = 1.0               # query dùng binary term presence
        S = (Q.astype(np.float32) @ self.W).toarray()   # (nq, n_chunk)
        k = min(topk, S.shape[1])
        ix = np.argpartition(-S, k - 1, axis=1)[:, :k]
        srt = np.take_along_axis(S, ix, 1).argsort(axis=1)[:, ::-1]
        ix = np.take_along_axis(ix, srt, 1)
        return ix, np.take_along_axis(S, ix, 1)

import time, pickle
t0 = time.time()
bm25 = SparseBM25().fit(chunks.text.tolist())
print('BM25 fit: %.1fs | vocab %d' % (time.time() - t0, bm25.W.shape[0]))

In [ ]:
#@title Đánh giá BM25 trên val
def retrieve_bm25(qids, qtext, batch=64):
    preds = {}
    for i in range(0, len(qids), batch):
        bq, bt = qids[i:i+batch], qtext[i:i+batch]
        ix, sc = bm25.search(bt)
        for q, row_ix, row_sc in zip(bq, ix, sc):
            preds[q] = chunks_to_docs(row_sc, row_ix)[0]
    return preds

t0 = time.time()
bm25_val = retrieve_bm25(VAL_QIDS, [QUESTIONS[q] for q in VAL_QIDS])
print('%.1fs' % (time.time() - t0))
m = show('BM25 chunk-level', bm25_val)
log_exp('bm25', m, f'chunk={CHUNK_CHARS} k1={BM25_K1} b={BM25_B}')

---
## Stage 5 — Dense retrieval

Bước nặng duy nhất. **Corpus là cố định và dùng chung cho cả train lẫn public test** →
encode **một lần**, tái dùng cho mọi thí nghiệm sau. Đây là điểm tiết kiệm lớn nhất trên Colab.

Encode theo **shard 50k chunk, ghi từng shard ra Drive ngay**. Session Colab chết ở phút
thứ 50 mà không có shard là mất trắng cả tiếng GPU; có shard thì chỉ chạy lại phần thiếu.

Vector lưu **fp16** (~500k × 768 × 2B ≈ 750MB) — vừa RAM, vừa VRAM T4, không cần faiss:
`q @ emb.T` trên GPU đã đủ nhanh và khỏi phải build index.

In [ ]:
#@title Encode corpus (sharded, resume được)
import numpy as np, os, glob, time, re, torch
from sentence_transformers import SentenceTransformer

MODEL_PATH = FT_BIENCODER_PATH if USE_FINETUNED else BIENCODER
TAG = re.sub(r'[^a-zA-Z0-9]+', '_', MODEL_PATH) + f'_c{CHUNK_CHARS}' + ('_smoke' if SMOKE else '')
EMB_DIR = f'{ART}/emb/{TAG}'
os.makedirs(EMB_DIR, exist_ok=True)
print('cache:', EMB_DIR)

def encode_corpus():
    texts = chunks.text.tolist()
    shards = [(i, min(i + ENC_SHARD, len(texts))) for i in range(0, len(texts), ENC_SHARD)]
    todo = [(a, b) for a, b in shards if not os.path.exists(f'{EMB_DIR}/s_{a:08d}.npy')]
    if todo:
        model = SentenceTransformer(MODEL_PATH, device='cuda' if torch.cuda.is_available() else 'cpu')
        model.max_seq_length = MAX_SEQ
        if torch.cuda.is_available():
            model.half()
        print(f'{len(todo)}/{len(shards)} shard cần encode', flush=True)
        for a, b in todo:
            t0 = time.time()
            v = model.encode(texts[a:b], batch_size=ENC_BATCH, normalize_embeddings=True,
                             convert_to_numpy=True, show_progress_bar=False)
            np.save(f'{EMB_DIR}/s_{a:08d}.npy', v.astype(np.float16))
            print(f'  shard {a}-{b}: {time.time()-t0:.0f}s', flush=True)
        del model
        torch.cuda.empty_cache()
    return np.concatenate([np.load(f) for f in sorted(glob.glob(f'{EMB_DIR}/s_*.npy'))])

t0 = time.time()
EMB = encode_corpus()
assert len(EMB) == len(chunks), f'emb {len(EMB)} != chunks {len(chunks)} — xoá {EMB_DIR} và encode lại'
print('EMB', EMB.shape, EMB.dtype, '| %.1f MB | %.1fs' % (EMB.nbytes / 1e6, time.time() - t0))

In [ ]:
#@title Dense search trên GPU
import torch, numpy as np
from sentence_transformers import SentenceTransformer

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
EMB_T = torch.from_numpy(EMB).to(DEV)                       # (N, 768) fp16
qmodel = SentenceTransformer(MODEL_PATH, device=DEV)
qmodel.max_seq_length = MAX_SEQ
if DEV == 'cuda':
    qmodel.half()

def dense_search(qtext, topk=CAND_CHUNKS, batch=32):
    """-> (indices, scores) của top chunk cho mỗi query."""
    ixs, scs = [], []
    for i in range(0, len(qtext), batch):
        q = qmodel.encode(qtext[i:i+batch], batch_size=batch, normalize_embeddings=True,
                          convert_to_tensor=True, show_progress_bar=False).to(EMB_T.dtype)
        s = q @ EMB_T.T                                     # cosine (đã normalize)
        v, ix = torch.topk(s, min(topk, s.shape[1]), dim=1)
        ixs.append(ix.cpu().numpy()); scs.append(v.float().cpu().numpy())
    return np.concatenate(ixs), np.concatenate(scs)

def retrieve_dense(qids, qtext):
    ix, sc = dense_search(qtext)
    return {q: chunks_to_docs(s, i)[0] for q, i, s in zip(qids, ix, sc)}

t0 = time.time()
val_text = [QUESTIONS[q] for q in VAL_QIDS]
dense_val = retrieve_dense(VAL_QIDS, val_text)
print('%.1fs' % (time.time() - t0))
m = show('dense' + ('-ft' if USE_FINETUNED else ''), dense_val)
log_exp('dense_ft' if USE_FINETUNED else 'dense', m, MODEL_PATH)

---
## Stage 6 — Hybrid fusion (RRF)

Dùng **Reciprocal Rank Fusion** thay vì cộng điểm có trọng số: điểm BM25 và điểm cosine
nằm trên hai thang hoàn toàn khác nhau, chuẩn hoá chúng luôn mong manh. RRF chỉ dùng
**hạng**, không cần tune trọng số, và bền hơn nhiều.

`score(d) = Σ 1 / (K + rank_i(d))`

In [ ]:
#@title RRF fuse
def rrf(*rankings, k=RRF_K, topn=CAND_DOCS):
    """Mỗi ranking: {qid: [doc_id]} -> {qid: [doc_id] đã fuse}."""
    out = {}
    for q in rankings[0]:
        s = {}
        for r in rankings:
            for rank, d in enumerate(r.get(q, [])):
                s[d] = s.get(d, 0.0) + 1.0 / (k + rank + 1)
        out[q] = [d for d, _ in sorted(s.items(), key=lambda x: -x[1])[:topn]]
    return out

hybrid_val = rrf(bm25_val, dense_val)
m = show('hybrid RRF', hybrid_val)
log_exp('hybrid_rrf', m, f'K={RRF_K}')

---
## Stage 7 — Cross-encoder rerank

**Đây là bước ăn điểm mạnh nhất.** Baseline cho `recall@100 = 0.87` nhưng `recall@5 = 0.486`
— khoảng cách đó chính là dư địa của reranking, và cross-encoder tồn tại để thu hẹp nó.

Cross-encoder đọc `(question, chunk)` cùng lúc nên hiểu quan hệ sâu hơn bi-encoder, nhưng
đắt gấp hàng nghìn lần → chỉ chấm `RERANK_TOP=50` doc, mỗi doc đại diện bởi **chunk tốt nhất**
theo dense score.

Corpus chỉ 8.532 văn bản nên toàn bộ 1.000 câu public test × 50 ứng viên = 50k cặp,
khoảng 5–10 phút trên T4. Hoàn toàn khả thi.

In [ ]:
#@title Chọn chunk đại diện cho mỗi doc ứng viên
import numpy as np
from collections import defaultdict

def best_chunk_per_doc(qids, qtext, cand):
    """{qid: {doc_id: chunk_index tốt nhất}} — dựa trên dense score."""
    ix, sc = dense_search(qtext, topk=CAND_CHUNKS)
    best = {}
    for q, row_ix, row_sc in zip(qids, ix, sc):
        want, top = set(cand[q]), {}
        for ci, s in zip(row_ix, row_sc):
            d = DOC_IDS[CHUNK2DOC[ci]]
            if d in want and s > top.get(d, (-1e30, 0))[0]:
                top[d] = (s, ci)
        # doc nào chỉ do BM25 tìm ra (không có chunk trong top dense) -> dùng chunk đầu
        for d in want - set(top):
            first = int(np.searchsorted(CHUNK2DOC, DOC2IX[d]))
            top[d] = (-1e30, first)
        best[q] = {d: ci for d, (_, ci) in top.items()}
    return best

CHUNK_TEXTS = chunks.text.tolist()
print('ok')

In [ ]:
#@title Rerank
from sentence_transformers import CrossEncoder
import numpy as np, time, torch

reranker = CrossEncoder(RERANKER, max_length=RERANK_MAXLEN,
                        device='cuda' if torch.cuda.is_available() else 'cpu')

def rerank(qids, qtext, cand, top=RERANK_TOP):
    rep = best_chunk_per_doc(qids, qtext, {q: cand[q][:top] for q in qids})
    out = {}
    for n, (q, qt) in enumerate(zip(qids, qtext)):
        docs = cand[q][:top]
        if not docs:
            out[q] = []; continue
        pairs = [[qt, CHUNK_TEXTS[rep[q][d]]] for d in docs]
        s = reranker.predict(pairs, batch_size=RERANK_BATCH, show_progress_bar=False)
        order = np.argsort(-np.asarray(s))
        # doc ngoài top rerank vẫn giữ nguyên thứ tự cũ ở phía sau
        out[q] = [docs[i] for i in order] + cand[q][top:]
        if (n + 1) % 100 == 0:
            print(f'  {n+1}/{len(qids)}', flush=True)
    return out

t0 = time.time()
rerank_val = rerank(VAL_QIDS, val_text, hybrid_val)
print('%.1fs' % (time.time() - t0))
m = show('hybrid + rerank', rerank_val)
log_exp('hybrid_rerank', m, f'{RERANKER} top{RERANK_TOP}')

---
## Stage 8 — Fine-tune bi-encoder *(tuỳ chọn, ROI cao nhất sau reranker)*

6.000 cặp (câu hỏi → văn bản gold) trong train split là tài sản lớn nhất bạn có.
Embedding off-the-shelf hiểu tiếng Việt phổ thông nhưng **không** hiểu văn phong pháp quy.

Quy trình **iterative hard negative mining**:
1. Chạy retrieval hiện tại trên train split → top-50 doc.
2. Bỏ gold ra → phần còn lại là **hard negative** (giống gold nhưng sai — đúng thứ model
   cần học phân biệt; random negative gần như vô dụng).
3. Train `MultipleNegativesRankingLoss`.
4. Mine lại bằng model mới → train vòng 2. Thường thêm được vài điểm nữa.

Sau khi train xong: đặt `USE_FINETUNED = True` ở Stage 0 rồi **chạy lại Stage 5** (encode
lại toàn corpus bằng model mới). Vì vậy hãy train xong hẳn rồi mới encode lại — đừng
encode lại sau mỗi epoch.

> Batch càng lớn MNRL càng mạnh (in-batch negatives). T4 16GB chịu được batch 64–96 ở
> 256 token với fp16. Nếu OOM thì giảm `FT_BATCH`, đừng giảm `max_seq_length`.

In [ ]:
#@title Mine hard negatives + train
FT_BATCH   = 64
FT_MAXSEQ  = 256   # hạ so với MAX_SEQ để đổi lấy batch lớn hơn
FT_EPOCHS  = 2
DO_FINETUNE = False   #@param — bật khi muốn train

if DO_FINETUNE:
    import random, torch, numpy as np
    from sentence_transformers import SentenceTransformer, InputExample, losses
    from torch.utils.data import DataLoader

    tr_qids = TRAIN_QIDS
    tr_text = [QUESTIONS[q] for q in tr_qids]
    print('mine hard negatives...', flush=True)
    cand = rrf(retrieve_bm25(tr_qids, tr_text), retrieve_dense(tr_qids, tr_text))
    rep  = best_chunk_per_doc(tr_qids, tr_text, cand)

    # chunk đại diện của chính gold doc, dùng làm positive
    gold_rep = best_chunk_per_doc(tr_qids, tr_text, {q: list(GOLD[q]) for q in tr_qids})

    rng = random.Random(SEED)
    examples = []
    for q in tr_qids:
        gold = GOLD[q]
        pos_ci = [gold_rep[q][g] for g in gold if g in gold_rep[q]]
        if not pos_ci:
            continue
        negs = [d for d in cand[q][:50] if d not in gold]
        if not negs:
            continue
        for nd in rng.sample(negs, min(4, len(negs))):
            examples.append(InputExample(texts=[QUESTIONS[q],
                                                CHUNK_TEXTS[rng.choice(pos_ci)],
                                                CHUNK_TEXTS[rep[q][nd]]]))
    print(f'{len(examples)} training triplet')

    model = SentenceTransformer(BIENCODER, device='cuda')
    model.max_seq_length = FT_MAXSEQ
    dl = DataLoader(examples, shuffle=True, batch_size=FT_BATCH, drop_last=True)
    model.fit(train_objectives=[(dl, losses.MultipleNegativesRankingLoss(model))],
              epochs=FT_EPOCHS, warmup_steps=int(0.1 * len(dl) * FT_EPOCHS),
              use_amp=True, output_path=FT_BIENCODER_PATH, show_progress_bar=True)
    print('đã lưu ->', FT_BIENCODER_PATH)
    print('BƯỚC TIẾP: đặt USE_FINETUNED=True ở Stage 0, rồi chạy lại Stage 5 (encode lại corpus).')
else:
    print('DO_FINETUNE=False — bỏ qua.')

---
## Stage 9 — Dự đoán public test + validate + đóng gói

Validator chặn đúng những lỗi làm mất điểm oan:
- Thiếu qid so với `public-official.json`
- Câu có **> 5** id → BTC cho câu đó **0 điểm cả Recall lẫn Precision**
- Câu có **< 5** id → tự bỏ điểm Recall vô cớ (Recall là metric chính, Precision chỉ tie-break)
- id trùng lặp trong một câu, id không phải string, id không tồn tại trong corpus
- `submission.json` bị nằm trong thư mục con của zip (lỗi rất hay gặp → rớt bài nộp)

In [ ]:
#@title Chạy pipeline trên public test
import time

def full_pipeline(qids, qtext):
    b = retrieve_bm25(qids, qtext)
    d = retrieve_dense(qids, qtext)
    h = rrf(b, d)
    return rerank(qids, qtext, h)

pub_qids = list(PUBLIC.keys())
pub_text = [PUBLIC[q] for q in pub_qids]
t0 = time.time()
pub_preds = full_pipeline(pub_qids, pub_text)
print('%d câu | %.1f phút' % (len(pub_qids), (time.time() - t0) / 60))

In [ ]:
#@title Validate + zip
import json, zipfile, os, collections

def build_submission(preds, all_qids, corpus_ids):
    """Luôn trả về ĐÚNG 5 id/câu. Thiếu thì bù bằng ứng viên tốt nhất còn lại."""
    sub, filler = {}, [d for d in DOC_IDS[:TOPK_SUBMIT]]
    for q in all_qids:
        ids, seen = [], set()
        for d in preds.get(q, []):
            d = str(d)
            if d not in seen and d in corpus_ids:
                seen.add(d); ids.append(d)
            if len(ids) == TOPK_SUBMIT:
                break
        for f in filler:                       # câu không có ứng viên -> vẫn đủ 5
            if len(ids) == TOPK_SUBMIT:
                break
            if f not in seen:
                seen.add(f); ids.append(f)
        sub[q] = {'answer': ids}
    return sub

def validate(sub, all_qids, corpus_ids):
    errs = []
    # scoring.py của BTC: len(ids_preds) != len(ids_truth) -> raise Exception
    # -> lệch số câu làm SẬP TOÀN BỘ bài nộp, không phải chỉ mất điểm 1 câu.
    if len(sub) != len(all_qids):
        errs.append(f'SẬP TOÀN BÀI: {len(sub)} câu nộp vs {len(all_qids)} câu đề')
    if set(sub) != set(all_qids):
        errs.append(f'SẬP TOÀN BÀI: thiếu {len(set(all_qids)-set(sub))} qid, '
                    f'thừa {len(set(sub)-set(all_qids))} qid (len(None) -> TypeError)')
    for q, v in sub.items():
        a = v.get('answer')
        if not isinstance(a, list):
            errs.append(f'{q}: answer không phải list'); continue
        if len(a) == 0:
            errs.append(f'{q}: answer rỗng -> câu này 0 điểm')
        if len(a) > TOPK_SUBMIT:
            errs.append(f'{q}: {len(a)} id > {TOPK_SUBMIT} -> CÂU NÀY BỊ 0 ĐIỂM')
        if len(a) < TOPK_SUBMIT:
            errs.append(f'{q}: chỉ {len(a)} id -> mất điểm Recall vô cớ')
        if len(set(a)) != len(a):
            errs.append(f'{q}: có id trùng lặp')
        if any(not isinstance(x, str) for x in a):
            errs.append(f'{q}: id phải là string')
        if any(x not in corpus_ids for x in a):
            errs.append(f'{q}: có id không tồn tại trong corpus')
    return errs

assert not SMOKE, 'SMOKE=True — đây là kết quả trên corpus thu nhỏ, KHÔNG được nộp. Đặt SMOKE=False và chạy lại.'

corpus_ids = set(corpus.doc_id)
sub = build_submission(pub_preds, list(PUBLIC), corpus_ids)
errs = validate(sub, list(PUBLIC), corpus_ids)

if errs:
    print(f'{len(errs)} LỖI — chưa nộp được:')
    for e in errs[:20]:
        print('  -', e)
else:
    out_json = f'{SCRATCH}/submission.json'
    out_zip  = f'{ART}/submission_{PROFILE}.zip'
    with open(out_json, 'w', encoding='utf-8') as f:
        json.dump(sub, f, ensure_ascii=False)
    with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(out_json, arcname='submission.json')   # arcname -> nằm ở GỐC zip
    with zipfile.ZipFile(out_zip) as z:
        assert z.namelist() == ['submission.json'], z.namelist()
    print('OK ->', out_zip, '(%.0f KB)' % (os.path.getsize(out_zip) / 1e3))
    print('phân bố số id/câu:', collections.Counter(len(v['answer']) for v in sub.values()))
    print(json.dumps(dict(list(sub.items())[:2]), ensure_ascii=False, indent=1))

---
## Stage 10 — Xem lại log thí nghiệm

In [ ]:
#@title Bảng thí nghiệm
import pandas as pd, os
if os.path.exists(LOG_CSV):
    df = pd.read_csv(LOG_CSV)
    cols = [c for c in ['time','profile','run','smoke','OFFICIAL_recall',
                        'recall@5','recall@20','recall@100','note'] if c in df.columns]
    display(df[cols].tail(25))
else:
    print('chưa có log')

---
## Việc cần làm ngoài notebook

**1. Đăng ký model với BTC — bắt buộc, hạn 18/09.**
Bài nộp dùng model chưa được phê duyệt sẽ **không được công nhận**.
[Form đăng ký](https://forms.gle/HWE7tcxzWq63Kxv28) ·
[Danh sách đã duyệt](https://docs.google.com/spreadsheets/d/1c5jzsYezWho1WGLRfMKWOaFPLIk_GTnXP5vV8AOWM2Q/edit)

Notebook này dùng: `bkai-foundation-models/vietnamese-bi-encoder` (135M) +
`BAAI/bge-reranker-v2-m3` (568M) = **0.70B < 4B**. Nên đăng ký rộng hơn thực dùng ngay
từ đầu (thêm vài phương án dự phòng) vì chờ duyệt mất thời gian — và nếu bạn chạy cả hai
notebook thì phải đăng ký **cả 4 model**.

**2. Nộp bằng đúng Organization `SGU_LegalMind`** trên Codabench. Bài nộp không qua
Organization là **không hợp lệ**.

**3. Ngân sách 10 lượt nộp/ngày.** Tune hoàn toàn trên val offline, chỉ nộp khi val có cải
thiện thật. 1.000 câu public là tập nhỏ — chênh 1–2% rất có thể chỉ là nhiễu, đừng
overfit leaderboard, vì private test mới là điểm ăn thua.

## Thứ tự chạy đề xuất

| # | Việc | GPU? | Kỳ vọng |
|---|---|---|---|
| 1 | `SMOKE=True`, chạy hết notebook | không | pipeline không lỗi |
| 2 | `SMOKE=False`, Stage 1–4 (BM25) | **không** | so với TF-IDF `recall@5=0.486` |
| 3 | Stage 5 encode + dense | có | `recall@100` nên > 0.90 |
| 4 | Stage 6 RRF | có | tăng nhẹ so với từng nhánh |
| 5 | Stage 7 rerank → **nộp lần đầu** | có | bước tăng mạnh nhất |
| 6 | Stage 8 fine-tune → encode lại → nộp | có | bước tăng lớn thứ hai |

Chạy bước 2 trong session **CPU** để giữ quota GPU — đó là tài nguyên khan hiếm nhất
trong 38 ngày còn lại.

## Nếu muốn đẩy xa hơn
- **Fine-tune cả reranker** trên train split (thường hơn fine-tune bi-encoder).
- **Ensemble** nhiều bi-encoder (e5, gte-multilingual) rồi RRF — vẫn dư ngân sách 4B.
- **Query expansion**: câu hỏi chỉ ~20 từ, rất ngắn.
- **Doc-level prior**: 3.105/8.532 văn bản từng là gold trong train — phân bố chủ đề của
  corpus có thể khai thác được, nhưng cẩn thận overfit.
